# Phase 1: 3-way 구조 검증

**핵심 질문**: 모델을 E(공유 인코더) + θ_task(공유 태스크 헤드) + θ_BS(사이트 임베딩)로 3-way 분리하는 것이 기존 방법보다 좋은가?

Phase 0에서 ReEsNet 채널 추정이 우리 데이터에서 정상 동작함을 확인했다.
Phase 1에서는 이 ReEsNet을 **여러 BS가 협력 학습하는 FL 환경**에 넣고, 3-way 분리 구조의 우위를 검증한다.

이 Phase가 실패하면 논문의 핵심 contribution이 무너지므로, 가장 중요한 실험이다.

### 실험 구성
- **Pre-train BS**: BS 0~5 (학습에 참여)
- **Test BS**: BS 6, 7 (학습에 참여하지 않은 unseen 사이트)

### Training
```bash
python -m src.experiments.1_fl_verification.train --step all --gpus 0 1 2 3
```

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.dataset_operation.dataset import ChannelEstimationDataset
from src.models.estimator import create_model
from src.models.baselines import PlainEstimator, FedPerEstimator
from src.training.trainer import evaluate, evaluate_per_snr, load_checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
DATA_DIR = Path(f'{PROJECT_ROOT}/assets/data/channels').resolve()
CKPT_DIR = Path(f'{PROJECT_ROOT}/assets/checkpoints/phase1')
ALL_BS = list(range(8))
PRETRAIN_BS = [0, 1, 2, 3, 4, 5]
TEST_BS = [6, 7]
METHODS = ['fedavg', 'fedper', '3way']

---
## Step 1-1: FL 방법론 비교 (3-way vs 2-way vs FedAvg vs Independent)

### 의의
이 실험은 **3-way 분리 구조의 존재 이유**를 증명하는 가장 기본적인 실험이다.

기존 FL에서는 모델을 2가지로 나눈다:
- **FedAvg**: 모델 전체를 공유 → BS별 특성을 반영 못함
- **FedPer (2-way)**: 인코더만 공유 + 태스크 헤드는 로컬 → 어느 정도 개인화

우리의 **3-way**는 여기에 θ_BS(사이트 임베딩)를 추가한다:
- E(인코더) + θ_task(태스크 헤드) = 공유 → 모든 BS에서 학습한 무선 물리 지식을 최대한 활용
- θ_BS = 로컬 → 각 BS의 고유한 위치/환경 특성만 ~64차원 벡터로 압축

### 비교 대상

| 방법 | 서버에서 공유 | BS 로컬 | 특징 |
|------|-------------|---------|------|
| **3-way (Ours)** | E + θ_task | θ_BS (~64d) | 사이트 특성만 분리 |
| **FedPer (2-way)** | E | θ_task | 헤드 전체가 로컬 → 공유 지식 적음 |
| **FedAvg** | 전부 | 없음 | 개인화 없음 |
| **Independent** | 없음 | 전부 | 협력 없음, 각자 학습 |

### 성공 기준
- 3-way ≥ FedPer > FedAvg ≥ Independent (per-BS NMSE)
- 3-way가 FedPer보다 최소 **1 dB** 이상 개선

In [3]:
# Build val loaders
val_loaders = {}
for bs_id in ALL_BS:
    ds = ChannelEstimationDataset(data_dir=DATA_DIR, bs_ids=[bs_id], snr_range_db=(0, 30))
    n_val = max(int(len(ds) * 0.2), 1)
    _, val_ds = torch.utils.data.random_split(ds, [len(ds) - n_val, n_val])
    val_loaders[bs_id] = DataLoader(val_ds, batch_size=128)
    print(f'BS{bs_id}: {n_val} val samples')

  Filtered 3/1276 NaN samples
BS0: 254 val samples
  Filtered 37/1421 NaN samples
BS1: 276 val samples
  Filtered 121/1241 NaN samples
BS2: 224 val samples
  Filtered 7/1152 NaN samples
BS3: 229 val samples
  Filtered 2/1102 NaN samples
BS4: 220 val samples
BS5: 269 val samples
BS6: 232 val samples
BS7: 259 val samples


In [4]:
# Load per-BS models and evaluate
model_factories = {
    'fedavg': lambda: PlainEstimator(encoder_channels=64, encoder_blocks=3),
    'fedper': lambda: FedPerEstimator(encoder_channels=64, encoder_blocks=3),
    '3way': lambda: create_model(site_integration='film', site_embed_dim=64),
    'independent': lambda: PlainEstimator(encoder_channels=64, encoder_blocks=3),
}

final_nmse = {}
for method in METHODS + ['independent']:
    final_nmse[method] = {}
    for bs_id in ALL_BS:
        if method == 'independent':
            ckpt_name = f'phase0/reesnet_bs{bs_id}_independent'
        else:
            ckpt_name = f'phase1/1-1_{method}_bs{bs_id}'
        try:
            m = model_factories[method]().to(device)
            load_checkpoint(m, ckpt_name, device=device)
            _, db = evaluate(m, val_loaders[bs_id], device)
            final_nmse[method][bs_id] = db
        except FileNotFoundError:
            print(f'{method} BS{bs_id}: not found')
            final_nmse[method][bs_id] = None

print('\nPer-BS NMSE (dB):')
for method in final_nmse:
    vals = [f'{final_nmse[method][bs]:.1f}' if final_nmse[method][bs] is not None else 'N/A'
            for bs in ALL_BS]
    print(f'  {method:12s}: {vals}')

  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedavg_bs0.pt
  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedavg_bs1.pt
  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedavg_bs2.pt
  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedavg_bs3.pt
  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedavg_bs4.pt
  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedavg_bs5.pt
  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedavg_bs6.pt
  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedavg_bs7.pt
  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedper_bs0.pt
  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedper_bs1.pt
  Loaded: /home/sungmin/Projects/6g-collaborative_bs/checkpoints/phase1/1-1_fedper_bs2.pt
  Loaded: 

In [ ]:
if histories:
    fig, ax = plt.subplots(figsize=(8, 5))
    for method, hist in histories.items():
        avg_per_round = [
            np.mean([hist['val_nmse_db'][str(bs)][r]
                     for bs in ALL_BS if hist['val_nmse_db'].get(str(bs)) and
                     hist['val_nmse_db'][str(bs)][r] is not None])
            for r in range(len(hist['round']))
        ]
        ax.plot(avg_per_round, label=method, linewidth=2)
    ax.set_xlabel('FL Round')
    ax.set_ylabel('Avg NMSE (dB)')
    ax.set_title('FL Convergence Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-BS bar chart
display_names = {'3way': '3-way (Ours)', 'fedper': 'FedPer', 'fedavg': 'FedAvg', 'independent': 'Independent'}
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(ALL_BS))
width = 0.2
for i, method in enumerate(['3way', 'fedper', 'fedavg', 'independent']):
    vals = [final_nmse[method].get(bs, 0) or 0 for bs in ALL_BS]
    ax.bar(x + i*width, vals, width, label=display_names[method])

ax.set_xlabel('BS ID')
ax.set_ylabel('NMSE (dB)')
ax.set_title('Per-BS Channel Estimation Performance')
ax.set_xticks(x + 1.5*width)
ax.set_xticklabels([f'BS{i}' for i in ALL_BS])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Check criterion
for bs in ALL_BS:
    vals = {m: final_nmse[m].get(bs) for m in ['3way', 'fedper', 'fedavg', 'independent']}
    if all(v is not None for v in vals.values()):
        order_ok = vals['3way'] <= vals['fedper'] <= vals['fedavg']
        print(f'BS{bs}: 3way={vals["3way"]:.1f}, fedper={vals["fedper"]:.1f}, '
              f'fedavg={vals["fedavg"]:.1f}, indep={vals["independent"]:.1f} '
              f'[{"OK" if order_ok else "UNEXPECTED"}]')

---
## Step 1-2: Few-Shot Adaptation (k-shot)

### 의의
이 실험은 3-way 구조의 **실용적 가치**를 증명한다.

현실 시나리오: 새로운 기지국(BS 6, 7)을 배치했는데, 데이터가 아직 거의 없다 (cold start).
이때 기존 BS 0~5에서 사전학습한 모델을 가져와서, **소량의 데이터(k개)만으로 빠르게 적응**할 수 있는가?

핵심은 θ_BS의 효율성이다:
- **θ_BS-only**: 사전학습된 E + θ_task는 freeze, θ_BS(~64개 파라미터)만 업데이트 → 소량 데이터에 overfitting 위험 적음
- **Fine-tune all**: 모델 전체를 업데이트 → 파라미터 많아서 소량 데이터에 overfitting
- **From scratch**: 사전학습 없이 처음부터 → k가 작으면 학습 자체가 안 됨
- **MAML**: meta-learning 초기화 → 비교용 baseline

k={5, 10, 20, 50, 100, 200}으로 실험하고, 각 k에서 N=5 반복으로 error bar를 구한다.

### 성공 기준
- k < 50에서 θ_BS-only가 다른 방법보다 **NMSE 2+ dB** 우세
- k가 커지면 fine-tune-all이 따라잡는 것은 정상 (θ_BS의 가치는 cold start 구간)

In [7]:
# Load few-shot results
results_path = CKPT_DIR / '1-2_fewshot_results.json'
if results_path.exists():
    with open(results_path) as f:
        fewshot = json.load(f)
    print(f'Loaded {len(fewshot)} result groups')
else:
    print(f'Not found: {results_path}')
    print('Run: python -m src.experiments.1_fl_verification.train --step 1-2')
    fewshot = {}

Loaded 8 result groups


In [ ]:
if fewshot:
    for test_bs in TEST_BS:
        fig, ax = plt.subplots(figsize=(8, 5))
        method_map = {
            'theta_bs_only': ('theta_BS-only (Ours)', 'C0'),
            'finetune_all': ('Fine-tune all', 'C1'),
            'from_scratch': ('From scratch', 'C2'),
            'maml': ('MAML', 'C3'),
        }
        for method, (label, color) in method_map.items():
            key = f'bs{test_bs}_{method}'
            if key not in fewshot:
                continue
            k_vals = sorted([int(k) for k in fewshot[key].keys()])
            means = [np.mean(fewshot[key][str(k)]) for k in k_vals]
            stds = [np.std(fewshot[key][str(k)]) for k in k_vals]
            ax.errorbar(k_vals, means, yerr=stds, marker='o', label=label,
                        color=color, capsize=3, linewidth=2)

        ax.set_xlabel('Number of adaptation samples (k)')
        ax.set_ylabel('NMSE (dB)')
        ax.set_title(f'Few-Shot Adaptation to BS{test_bs}')
        ax.set_xscale('log')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()

        # Check criterion
        key_theta = f'bs{test_bs}_theta_bs_only'
        key_ft = f'bs{test_bs}_finetune_all'
        key_maml = f'bs{test_bs}_maml'
        if key_theta in fewshot and key_ft in fewshot:
            for k in [5, 10, 20]:
                if str(k) in fewshot[key_theta] and str(k) in fewshot[key_ft]:
                    diff_ft = np.mean(fewshot[key_ft][str(k)]) - np.mean(fewshot[key_theta][str(k)])
                    msg = f'  BS{test_bs} k={k}: theta_BS-only vs fine-tune = {diff_ft:.2f} dB'
                    if key_maml in fewshot and str(k) in fewshot[key_maml]:
                        diff_maml = np.mean(fewshot[key_maml][str(k)]) - np.mean(fewshot[key_theta][str(k)])
                        msg += f', vs MAML = {diff_maml:.2f} dB'
                    print(msg + f' [{"PASS" if diff_ft > 2 else "CHECK"}]')

---
## Step 1-3: Pre-trained E vs From-Scratch 수렴 속도

### 의의
이 실험은 **공유 인코더 E가 학습한 지식이 실제로 전이 가능한지** 확인한다.

Step 1-2가 "소량 데이터에서 얼마나 잘하나"였다면, Step 1-3는 "충분한 데이터가 있을 때 **얼마나 빨리 수렴하나**"를 본다.

비교:
- **Pre-trained**: BS 0~5에서 학습한 E + θ_task를 초기값으로 사용 → 새 BS에서 학습 시작
- **From scratch**: 랜덤 초기화에서 시작 → 같은 데이터로 학습

E가 무선 채널의 보편적 물리 특성(path loss, multipath fading 등)을 잘 배웠다면, 새 BS에서도 이 지식이 유효하므로 수렴이 훨씬 빨라야 한다. 반대로 E가 학습 BS에 overfitting되었다면 전이 효과가 미미할 것.

### 성공 기준
- Pre-trained가 같은 NMSE 도달까지 **수렴 속도 3배 이상** 빠름
- 예: from-scratch가 60 epoch 걸리는 성능을 pre-trained는 20 epoch 이내에 달성

In [ ]:
for test_bs in TEST_BS:
    print(f'\n--- BS{test_bs} ---')
    methods = {}

    m_pre = create_model(site_integration='film', site_embed_dim=64).to(device)
    try:
        meta_pre = load_checkpoint(m_pre, f'phase1/1-3_pretrained_bs{test_bs}', device=device)
        methods['pretrained'] = meta_pre
        print(f'  Pretrained: {meta_pre.get("best_val_db", "N/A")} dB')
    except FileNotFoundError:
        print(f'  Pretrained: not found')

    m_scratch = PlainEstimator(encoder_channels=64, encoder_blocks=3).to(device)
    try:
        meta_scratch = load_checkpoint(m_scratch, f'phase1/1-3_scratch_bs{test_bs}', device=device)
        methods['from_scratch'] = meta_scratch
        print(f'  From scratch: {meta_scratch.get("best_val_db", "N/A")} dB')
    except FileNotFoundError:
        print(f'  From scratch: not found')

    if methods and all('val_losses' in m for m in methods.values()):
        fig, ax = plt.subplots(figsize=(8, 5))
        for name, meta in methods.items():
            ax.plot(10*np.log10(meta['val_losses']), label=name, linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('NMSE (dB)')
        ax.set_title(f'Adaptation to Unseen BS{test_bs}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    elif methods:
        print('  val_losses not in checkpoint. Re-run: python -m src.experiments.1_fl_verification.train --step 1-3')